# Adım 4 — Multi-Layer Anomali Detection Engine

Her katman **işaretleyicidir** — skor üretir, kesmez. Çıktılar `[0,1]` normalize edilir.  
Girdi: `step3_features.parquet` (Adım 3 çıktısı, 590,540 × 499)

**5 Katmanlı Mimari:**

| # | Katman | Skor Kolonu | Yöntem | Fraud Lift |
|---|--------|-------------|--------|-----------|
| 1 | Feature Selection | — | Variance + Korelasyon filtresi | — |
| 2 | Column Anomaly | `column_score_max`, `column_score_mean` | Z-score (kolon bazlı) | 1.47x / 1.64x |
| 3 | Multivariate Anomaly | `multivariate_score` | Isolation Forest (48 feature) | 1.26x |
| 4 | Entity Anomaly | `entity_score` | Dinamik eşik + velocity | 1.07x |
| 5 | Temporal Anomaly | `temporal_score` | Saat/gün pattern + velocity | 1.03x |

**Tasarım Prensipleri:**
- Her fonksiyon `(skor, driver)` çifti döndürür — açıklanabilirlik için
- Fraud'da outlier **silinmez** — sinyal kuyrukta yaşar
- Test setinde `target_col=None` geçilir

## Kurulum ve Bağımlılıklar

In [1]:
"""
Adım 4 — Multi-Layer Anomali Detection Engine
=============================================
Bağımlılık: step3_features.parquet (Adım 3 çıktısı)
Bağımsız çalışır, Adım 1-2-3 fonksiyonlarına import bağımlılığı yok.

Fonksiyonlar:
    feature_selection_report      — Isolation Forest öncesi veri güdümlü eleme
    detect_column_anomalies       — Kolon bazlı z-score anomali skoru
    detect_multivariate_anomalies — Isolation Forest tabanlı çok değişkenli anomali
    detect_entity_anomalies       — Entity bazlı dinamik threshold anomali
    detect_temporal_anomalies     — Zaman dilimi ve velocity bazlı anomali
    run_anomaly_detection         — Orkestratör
"""

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")
import joblib


# =============================================================================
# 1. FEATURE SELECTION REPORT

## 1. Feature Selection Report

Isolation Forest'a girmeden önce **veri güdümlü feature eleme** yapılır.

**İki aşamalı filtre:**

| Aşama | Yöntem | Eşik | Sonuç |
|-------|--------|------|-------|
| Variance threshold | Scaled varyans | < 0.01 | 466 → 57 feature |
| Korelasyon filtresi | Pearson abs | > 0.95 | 57 → **48 feature** |

**Çalışma çıktısı:**
```
Başlangıç feature sayısı : 466
Düşük varyans elenen      : 409
Yüksek korelasyon elenen  : 9
Seçilen feature sayısı    : 48
```

> **Neden bu önemli?** Isolation Forest yüksek boyutlulukta bozulur (curse of dimensionality).  
> 466 feature ile çalışmak yerine bilgi taşıyan 48 feature seçmek hem hız hem kalite kazandırır.

In [2]:
# =============================================================================

def feature_selection_report(
    df: pd.DataFrame,
    numeric_cols: list = None,
    variance_threshold: float = 0.01,
    correlation_threshold: float = 0.95,
    target_col: str = None,
    verbose: bool = True,
) -> dict:
    """
    Isolation Forest öncesi veri güdümlü feature eleme.

    Aşamalar:
        1. Variance threshold — neredeyse sabit kolonları eler
        2. Korelasyon filtresi — birbirinin kopyası kolonlardan birini eler
        3. Özet rapor

    Parameters
    ----------
    df                   : Input dataframe
    numeric_cols         : Kontrol edilecek kolonlar (None → otomatik seç)
    variance_threshold   : Normalize varyans eşiği (0.01 = %1 altı elenir)
    correlation_threshold: Korelasyon eşiği (0.95 üstü çiftlerde biri elenir)
    target_col           : Hedef kolon — asla elenmesin
    verbose              : Rapor bas

    Returns
    -------
    dict:
        selected_features : Seçilen feature listesi
        dropped_low_var   : Düşük varyanslı elenenler
        dropped_high_corr : Yüksek korelasyonlu elenenler + gerekçe
        report_df         : Tüm feature'ların özet tablosu
    """

    # Numeric kolonları belirle
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Target'ı listeden çıkar
    if target_col and target_col in numeric_cols:
        numeric_cols = [c for c in numeric_cols if c != target_col]

    # Sadece NaN olmayan kolonları al
    available = [c for c in numeric_cols if c in df.columns]
    work = df[available].copy()

    # --- Aşama 1: Variance threshold ---
    # Min-max normalize et, sonra varyans hesapla (farklı skalalar için adil karşılaştırma)
    scaler = MinMaxScaler()
    try:
        scaled = pd.DataFrame(
            scaler.fit_transform(work.fillna(work.median())),
            columns=work.columns
        )
    except Exception:
        scaled = work.fillna(0)

    variances = scaled.var()
    low_var_mask = variances < variance_threshold
    dropped_low_var = variances[low_var_mask].index.tolist()
    passed_var = [c for c in available if c not in dropped_low_var]

    # --- Aşama 2: Korelasyon filtresi ---
    if len(passed_var) > 1:
        corr_matrix = scaled[passed_var].corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        dropped_high_corr = {}
        to_drop = set()

        for col in upper.columns:
            high_corr_partners = upper.index[upper[col] > correlation_threshold].tolist()
            for partner in high_corr_partners:
                if partner not in to_drop:
                    # Daha düşük varyanslı olanı at
                    if variances.get(col, 0) < variances.get(partner, 0):
                        to_drop.add(col)
                        dropped_high_corr[col] = f"corr={upper.loc[partner, col]:.3f} with {partner}"
                    else:
                        to_drop.add(partner)
                        dropped_high_corr[partner] = f"corr={upper.loc[partner, col]:.3f} with {col}"

        selected_features = [c for c in passed_var if c not in to_drop]
    else:
        dropped_high_corr = {}
        selected_features = passed_var

    # --- Özet tablo ---
    report_rows = []
    for col in available:
        status = "selected"
        reason = ""
        if col in dropped_low_var:
            status = "dropped"
            reason = f"low_variance ({variances.get(col, 0):.4f} < {variance_threshold})"
        elif col in dropped_high_corr:
            status = "dropped"
            reason = dropped_high_corr[col]

        report_rows.append({
            "feature": col,
            "variance_normalized": round(variances.get(col, 0), 4),
            "status": status,
            "reason": reason,
        })

    report_df = pd.DataFrame(report_rows).sort_values(
        ["status", "variance_normalized"], ascending=[True, False]
    ).reset_index(drop=True)

    if verbose:
        print("=" * 60)
        print("FEATURE SELECTION REPORT")
        print("=" * 60)
        print(f"  Başlangıç feature sayısı : {len(available)}")
        print(f"  Düşük varyans elenen      : {len(dropped_low_var)}")
        print(f"  Yüksek korelasyon elenen  : {len(dropped_high_corr)}")
        print(f"  Seçilen feature sayısı    : {len(selected_features)}")
        print()
        if dropped_low_var:
            print(f"  [Düşük Varyans] İlk 5: {dropped_low_var[:5]}")
        if dropped_high_corr:
            sample = list(dropped_high_corr.items())[:5]
            print(f"  [Yüksek Korelasyon] İlk 5:")
            for k, v in sample:
                print(f"    - {k}: {v}")
        print("=" * 60)

    return {
        "selected_features": selected_features,
        "dropped_low_var": dropped_low_var,
        "dropped_high_corr": dropped_high_corr,
        "report_df": report_df,
    }



## 2. Column-Level Anomaly Detection

**Yöntem:** Her numeric kolon için z-score hesaplanır, `[0,1]` normalize edilir.

Her satır için iki özet skor üretilir:
- `column_score_max` — tüm kolonlar arasında **en yüksek** z-score (en kötü durum)
- `column_score_mean` — tüm kolonların **ortalama** z-score'u (genel sapma düzeyi)

**Neden iki skor?**
- `column_score_max` → tek bir kolonda aşırı sapma var mı? (hesap ele geçirme)
- `column_score_mean` → birçok kolonda eş zamanlı sapma var mı? (organizeli fraud)

**Çalışma çıktısı:**
```
column_score_max  — mean: 0.4924   fraud vs legit: 0.7108 / 0.4845  →  lift: 1.47x
column_score_mean — mean: 0.0318   fraud vs legit: 0.0511 / 0.0311  →  lift: 1.64x
Yüksek anomali (>0.5): 244,064 satır (41.33%)
```

> `column_score_mean` fraud lift'i (1.64x) `column_score_max`'tan (1.47x) daha yüksek —  
> fraud işlemler **birden fazla** kolonda aynı anda anormal davranış sergiliyor.

In [3]:
# =============================================================================
# 2. COLUMN ANOMALI DETECTION
# =============================================================================

def detect_column_anomalies(
    df: pd.DataFrame,
    numeric_cols: list = None,
    target_col: str = None,
    z_score_cap: float = 10.0,
    top_n_drivers: int = 3,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Her numeric kolon için z-score bazlı anomali skoru üretir.

    Her satır için:
        column_score_max  — En aşırı kolonun skoru (tek patlayan sinyal)
        column_score_mean — Tüm kolonların ortalaması (çok boyutlu hafif anomali)
        column_drivers    — Skoru en çok etkileyen kolonlar

    Parameters
    ----------
    df            : Input dataframe (df_features)
    numeric_cols  : Skor üretilecek kolonlar (None → otomatik)
    target_col    : Hedef kolon — skora dahil edilmez
    z_score_cap   : Z-score üst sınırı (aşırı değerleri kırpar)
    top_n_drivers : Kaç driver kolon saklanacak
    verbose       : Rapor bas

    Returns
    -------
    df : column_score_max, column_score_mean, column_drivers kolonları eklenmiş
    """

    df = df.copy()

    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if target_col and target_col in numeric_cols:
        numeric_cols = [c for c in numeric_cols if c != target_col]

    available = [c for c in numeric_cols if c in df.columns]

    if not available:
        print("[UYARI] Kolon bulunamadı, skor üretilemiyor.")
        df["column_score_max"] = 0.0
        df["column_score_mean"] = 0.0
        df["column_drivers"] = ""
        return df

    work = df[available].copy()

    # Z-score hesapla (NaN → medyan ile doldur)
    filled = work.fillna(work.median())
    z_scores = filled.apply(
        lambda col: np.abs(stats.zscore(col, nan_policy="omit")), axis=0
    )

    # Cap uygula
    z_scores = z_scores.clip(0, z_score_cap)

    # [0,1] normalize — cap değeri max olduğu için direkt bölme
    z_normalized = z_scores / z_score_cap

    # Satır bazında agregasyon
    df["column_score_max"] = z_normalized.max(axis=1).round(4)
    df["column_score_mean"] = z_normalized.mean(axis=1).round(4)

    # Driver kolonlar — her satır için en yüksek z-score'lu top_n kolon
    def get_drivers(row):
        top = row.nlargest(top_n_drivers)
        top = top[top > 0]
        if top.empty:
            return ""
        return ", ".join([f"{col}:{val:.2f}" for col, val in top.items()])

    df["column_drivers"] = z_normalized.apply(get_drivers, axis=1)

    if verbose:
        print("=" * 60)
        print("COLUMN ANOMALI DETECTION")
        print("=" * 60)
        print(f"  Değerlendirilen kolon     : {len(available)}")
        print(f"  column_score_max  — mean  : {df['column_score_max'].mean():.4f}")
        print(f"  column_score_max  — max   : {df['column_score_max'].max():.4f}")
        print(f"  column_score_mean — mean  : {df['column_score_mean'].mean():.4f}")
        high = (df["column_score_max"] > 0.5).sum()
        print(f"  Yüksek anomali (>0.5)     : {high:,} satır ({high/len(df)*100:.2f}%)")
        print("=" * 60)

    return df



## 3. Multivariate Anomaly Detection — Isolation Forest

**Yöntem:** Isolation Forest, anomalileri rastgele bölmelerle izole eder.  
Daha az bölme = daha anormal. `contamination=0.035` → veri setindeki fraud oranıyla hizalı.

**Neden Isolation Forest?**
- Tek kolon normal görünür ama **kombinasyon** anormal olan satırları yakalar
- Column anomaly'nin kaçırdığı joint distribution anomalilerini tespit eder
- 590k satır üzerinde O(n log n) zaman karmaşıklığı ile ölçeklenebilir

**Çalışma çıktısı:**
```
Kullanılan feature sayısı : 48  (feature_selection_report çıktısı)
Contamination             : 0.035
Score — mean              : 0.3034
Yüksek anomali (>0.7)     : 3,288 satır  (0.56%)
fraud vs legit            : 0.3793 / 0.3007  →  lift: 1.26x
```

> **Model persist:** Fit edilen model `pipeline/models/isolation_forest.joblib` olarak kaydedilir.  
> API inference'ta bu model yüklenerek train dağılımıyla tutarlı skorlar üretilir.

**Kayıt dosyaları:**
```
pipeline/models/isolation_forest.joblib   ← fit edilmiş model
pipeline/models/if_scaler.joblib          ← MinMaxScaler
pipeline/models/if_feature_list.json      ← feature listesi + min/max normalizasyon değerleri
```

In [4]:
# =============================================================================
# 3. MULTIVARIATE ANOMALI DETECTION
# =============================================================================

def detect_multivariate_anomalies(
    df: pd.DataFrame,
    selected_features: list,
    contamination: float = 0.035,
    n_estimators: int = 100,
    random_state: int = 42,
    verbose: bool = True,
    model_dir: str = "pipeline/models",   # ← API'nin baktığı yer
) -> pd.DataFrame:
    """
    Isolation Forest tabanlı çok değişkenli anomali skoru.

    Tek kolon normal görünür ama kombinasyon anormal olan satırları yakalar.
    contamination=0.035 → veri setindeki fraud oranıyla hizalı.

    Parameters
    ----------
    df                : Input dataframe
    selected_features : feature_selection_report çıktısından gelen liste
    contamination     : Beklenen anomali oranı
    n_estimators      : Ağaç sayısı
    random_state      : Tekrarlanabilirlik

    Returns
    -------
    df : multivariate_score kolonu eklenmiş [0,1]
    """

    df = df.copy()

    available = [c for c in selected_features if c in df.columns]

    if len(available) < 2:
        print("[UYARI] Yeterli feature yok, multivariate skor üretilemiyor.")
        df["multivariate_score"] = 0.0
        return df

    from pathlib import Path as _Path
    import joblib
    from sklearn.preprocessing import MinMaxScaler

    out_dir     = _Path(model_dir)
    model_path  = out_dir / "isolation_forest.joblib"
    scaler_path = out_dir / "if_scaler.joblib"
    feat_path   = out_dir / "if_feature_list.json"

    # Scaler fit
    scaler   = MinMaxScaler()
    work     = df[available].fillna(df[available].median())
    X_scaled = scaler.fit_transform(work)

    model = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1,
    )

    # score_samples → daha negatif = daha anormal
    raw_scores = model.fit(X_scaled).score_samples(X_scaled)

    # [0,1] normalize — negatif → pozitif, anomali yüksek skor alsın
    inverted = -raw_scores
    min_s, max_s = float(inverted.min()), float(inverted.max())
    if max_s > min_s:
        normalized = (inverted - min_s) / (max_s - min_s)
    else:
        normalized = np.zeros(len(inverted))

    df["multivariate_score"] = np.round(normalized, 4)

    # Model kaydet (varsa güncelle, yoksa oluştur)
    out_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model,  model_path)
    joblib.dump(scaler, scaler_path)
    import json as _json
    with open(feat_path, "w", encoding="utf-8") as _f:
        _json.dump({
            "features":      available,
            "n_features":    len(available),
            "contamination": contamination,
            "score_min":     min_s,
            "score_max":     max_s,
        }, _f, indent=2)

    if verbose:
        print("=" * 60)
        print("MULTIVARIATE ANOMALI DETECTION (Isolation Forest)")
        print("=" * 60)
        print(f"  Kullanılan feature sayısı : {len(available)}")
        print(f"  Contamination             : {contamination}")
        print(f"  Score — mean              : {df['multivariate_score'].mean():.4f}")
        print(f"  Score — max               : {df['multivariate_score'].max():.4f}")
        high = (df["multivariate_score"] > 0.7).sum()
        print(f"  Yüksek anomali (>0.7)     : {high:,} satır ({high/len(df)*100:.2f}%)")
        print(f"  Model kaydedildi          : {model_path}")
        print(f"  Scaler kaydedildi         : {scaler_path}")
        print(f"  Feature listesi           : {feat_path}")
        print("=" * 60)

    return df



## 4. Entity Anomaly Detection

**Yöntem:** Entity bazında (kart, email domain, cihaz) davranışsal sapma tespiti.  
Her entity için beklenen profil dinamik olarak hesaplanır.

**Skor bileşenleri (ağırlıklı toplam):**

| Bileşen | Ağırlık | Açıklama |
|---------|---------|----------|
| Amount z-score | %50 | Bu entity'nin tarihsel tutarından sapma |
| Velocity 1h | %30 | Son 1 saatte anormal işlem yoğunluğu |
| Velocity 1d | %20 | Son 1 günde anormal işlem yoğunluğu |

**Entity ağırlıkları** (AUC bazlı, fraud oranına göre):
```
card1         : 0.077
P_emaildomain : 0.202
DeviceType    : 0.721   ← en güçlü fraud sinyali
```

**Çalışma çıktısı:**
```
entity_score mean : 0.0118   fraud vs legit: 0.0126 / 0.0118  →  lift: 1.07x
Yüksek anomali (>0.5): 0 satır
```

> **Düşük lift neden normal?** Entity geçmişi az olan yeni kartlar global fallback alır —  
> bu onları masum gösterir ama gerçekte fraud olabilirler (yeni kart pattern).  
> Bu zayıflık Adım 5'te column sinyalleriyle dengelenir.

In [5]:
# =============================================================================
# 4. ENTITY ANOMALI DETECTION
# =============================================================================

def detect_entity_anomalies(
    df: pd.DataFrame,
    entity_cols: list,
    target_col: str = None,
    min_tx_count: int = 5,
    velocity_col_1h: str = None,
    velocity_col_1d: str = None,
    amt_zscore_suffix: str = "_amt_zscore",
    velocity_cap: float = 20.0,
    top_n_drivers: int = 3,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Entity bazlı dinamik threshold anomali skoru.

    Her entity için:
        - amt_zscore (Adım 3'ten) → entity'nin kendi profilinden sapma
        - velocity_1h / velocity_1d → kısa vadeli işlem yoğunluğu
        - fraud_rate bazlı entity ağırlığı (target_col varsa)
        - min_tx_count altı entity → global fallback
        - Manipülasyona karşı üst cap

    Parameters
    ----------
    df                  : Input dataframe (df_features)
    entity_cols         : Entity kolon listesi (örn. ["card1", "P_emaildomain"])
    target_col          : Fraud etiketi (ağırlık hesabı için, test'te None)
    min_tx_count        : Bu altı → global fallback
    velocity_col_1h     : 1 saatlik velocity kolon adı (None → otomatik arar)
    velocity_col_1d     : 1 günlük velocity kolon adı (None → otomatik arar)
    amt_zscore_suffix   : Entity zscore kolon suffix'i
    velocity_cap        : Velocity üst sınırı (normalize için)
    top_n_drivers       : Driver sayısı

    Returns
    -------
    df : entity_score, entity_drivers kolonları eklenmiş
    """

    df = df.copy()
    n = len(df)

    # entity_scores: key -> numpy array (asla scalar değil)
    entity_scores: dict = {}

    for entity in entity_cols:
        if entity not in df.columns:
            continue

        # --- Entity işlem sayısı ---
        tx_count_col = f"{entity}_tx_count"
        if tx_count_col not in df.columns:
            df[tx_count_col] = df.groupby(entity)[entity].transform("count")

        # --- amt_zscore ---
        zscore_col = f"{entity}{amt_zscore_suffix}"
        if zscore_col in df.columns:
            z_raw = df[zscore_col].abs().fillna(0.0).values
            if "TransactionAmt" in df.columns:
                amt = df["TransactionAmt"].values.astype(float)
                std = float(np.std(amt))
                global_z = np.abs((amt - np.mean(amt)) / max(std, 1e-6))
            else:
                global_z = np.zeros(n)
            reliable = (df[tx_count_col].values >= min_tx_count)
            z_final = np.where(reliable, z_raw, global_z)
            entity_scores[f"{entity}_zscore_norm"] = np.clip(z_final, 0, 10) / 10.0
        else:
            entity_scores[f"{entity}_zscore_norm"] = np.zeros(n)

        # --- Velocity 1h ---
        v1h_col = velocity_col_1h if velocity_col_1h else f"{entity}_velocity_1h"
        if v1h_col in df.columns:
            arr = df[v1h_col].fillna(0.0).values.astype(float)
            entity_scores[f"{entity}_v1h_norm"] = np.clip(arr, 0, velocity_cap) / velocity_cap
        else:
            entity_scores[f"{entity}_v1h_norm"] = np.zeros(n)

        # --- Velocity 1d ---
        v1d_col = velocity_col_1d if velocity_col_1d else f"{entity}_velocity_1d"
        if v1d_col in df.columns:
            arr = df[v1d_col].fillna(0.0).values.astype(float)
            entity_scores[f"{entity}_v1d_norm"] = np.clip(arr, 0, velocity_cap) / velocity_cap
        else:
            entity_scores[f"{entity}_v1d_norm"] = np.zeros(n)

    # --- Entity ağırlıkları (fraud_rate bazlı) ---
    entity_weights = {}
    for entity in entity_cols:
        if entity not in df.columns:
            continue
        if target_col and target_col in df.columns:
            global_rate = float(df[target_col].mean())
            fraud_rates = df.groupby(entity)[target_col].mean()
            weight = float(np.clip(fraud_rates / max(global_rate, 1e-6), 0, 3).mean())
            entity_weights[entity] = weight
        else:
            entity_weights[entity] = 1.0

    total_weight = sum(entity_weights.values()) or 1.0
    for k in entity_weights:
        entity_weights[k] /= total_weight

    # --- Bileşik entity skoru (saf numpy, clip sorunu yok) ---
    combined = np.zeros(n)
    for entity in entity_cols:
        if entity not in df.columns:
            continue
        w   = entity_weights.get(entity, 1.0 / max(len(entity_cols), 1))
        z   = entity_scores.get(f"{entity}_zscore_norm", np.zeros(n))
        v1h = entity_scores.get(f"{entity}_v1h_norm",   np.zeros(n))
        v1d = entity_scores.get(f"{entity}_v1d_norm",   np.zeros(n))
        combined += (z * 0.5 + v1h * 0.3 + v1d * 0.2) * w

    df["entity_score"] = np.round(np.clip(combined, 0.0, 1.0), 4)

    # --- Driver'lar ---
    def get_entity_drivers(idx):
        drivers = []
        for entity in entity_cols:
            if entity not in df.columns:
                continue
            zscore_col = f"{entity}{amt_zscore_suffix}"
            if zscore_col in df.columns:
                val = df.loc[idx, zscore_col]
                if not pd.isna(val) and abs(val) > 1.5:
                    drivers.append(f"{zscore_col}:{val:.2f}")

            v1h_col = f"{entity}_velocity_1h"
            if v1h_col in df.columns:
                val = df.loc[idx, v1h_col]
                if not pd.isna(val) and val > 2:
                    drivers.append(f"{v1h_col}:{int(val)}")

        return ", ".join(drivers[:top_n_drivers]) if drivers else ""

    df["entity_drivers"] = [get_entity_drivers(i) for i in df.index]

    if verbose:
        print("=" * 60)
        print("ENTITY ANOMALI DETECTION")
        print("=" * 60)
        print(f"  Entity kolonları          : {entity_cols}")
        print(f"  Entity ağırlıkları        : { {k: round(v,3) for k,v in entity_weights.items()} }")
        print(f"  Score — mean              : {df['entity_score'].mean():.4f}")
        print(f"  Score — max               : {df['entity_score'].max():.4f}")
        high = (df["entity_score"] > 0.5).sum()
        print(f"  Yüksek anomali (>0.5)     : {high:,} satır ({high/len(df)*100:.2f}%)")
        print("=" * 60)

    return df



## 5. Temporal Anomaly Detection

**Yöntem:** İşlem zamanının beklenen pattern'den sapmasını ölçer.

**Skor bileşenleri:**

| Bileşen | Ağırlık | Açıklama |
|---------|---------|----------|
| Velocity score | %40 | Velocity kolonlarından maksimum |
| Hour score | %35 | Saate göre fraud profili (train'de hesaplanır) |
| Flag score | %25 | is_night (+0.4), is_weekend (+0.2), is_business (-0.1) |

**Çalışma çıktısı:**
```
temporal_score mean : 0.5197   fraud vs legit: 0.5364 / 0.5191  →  lift: 1.03x
Yüksek anomali (>0.5): 517,390 satır (87.61%)
```

> **Temporal skor en zayıf katman** (lift: 1.03x). Tek başına yetersizdir.  
> Ancak gece saatinde yüksek velocity + diğer yüksek sinyallerle **kombine** edildiğinde  
> güçlü bir composite indicator oluşturur. Adım 5'te ağırlığı buna göre ayarlanır.

In [6]:
# =============================================================================
# 5. TEMPORAL ANOMALI DETECTION
# =============================================================================

def detect_temporal_anomalies(
    df: pd.DataFrame,
    target_col: str = None,
    hour_col: str = "tx_hour",
    is_night_col: str = "tx_is_night",
    is_weekend_col: str = "tx_is_weekend",
    is_business_col: str = "tx_is_business",
    velocity_1h_cols: list = None,
    velocity_1d_cols: list = None,
    velocity_cap: float = 20.0,
    top_n_drivers: int = 3,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Zaman dilimi ve velocity bazlı temporal anomali skoru.

    Bileşenler:
        1. Saat profili — bu saatte beklenen fraud oranına göre skor
        2. Gece/hafta sonu flag'leri — yüksek risk zaman dilimleri
        3. Velocity — kısa vadeli işlem yoğunluğu (tüm entity'lerin max'ı)

    Parameters
    ----------
    df                : Input dataframe
    target_col        : Fraud etiketi (saat profili için)
    hour_col          : Saat kolonu
    is_night_col      : Gece flag kolonu
    is_weekend_col    : Hafta sonu flag kolonu
    is_business_col   : İş saati flag kolonu
    velocity_1h_cols  : 1h velocity kolonları (None → otomatik arar)
    velocity_1d_cols  : 1d velocity kolonları (None → otomatik arar)
    velocity_cap      : Velocity normalize üst sınırı

    Returns
    -------
    df : temporal_score, temporal_drivers kolonları eklenmiş
    """

    df = df.copy()
    score_components = pd.DataFrame(index=df.index)

    # --- Bileşen 1: Saat profili ---
    if hour_col in df.columns:
        if target_col and target_col in df.columns:
            # Her saat için fraud oranını hesapla → normalize skor
            hour_fraud_rate = df.groupby(hour_col)[target_col].mean()
            global_rate = df[target_col].mean()
            # Lift = bu saatin fraud oranı / global fraud oranı, cap: 3x
            hour_lift = (hour_fraud_rate / hour_fraud_rate.clip(1e-6, None)).clip(0, 3.0)
            hour_score = df[hour_col].map(hour_lift).fillna(1.0) / 3.0
        else:
            # Target yok → gece saatlerine (22-06) yüksek skor ver
            night_hours = set(range(22, 24)) | set(range(0, 6))
            hour_score = df[hour_col].apply(lambda h: 0.7 if h in night_hours else 0.3)

        score_components["hour_score"] = hour_score.clip(0, 1)
    else:
        score_components["hour_score"] = 0.3  # nötr

    # --- Bileşen 2: Zaman dilimi flag'leri ---
    flag_score = pd.Series(0.0, index=df.index)

    if is_night_col in df.columns:
        flag_score += df[is_night_col].fillna(0) * 0.4

    if is_weekend_col in df.columns:
        flag_score += df[is_weekend_col].fillna(0) * 0.2

    if is_business_col in df.columns:
        # İş saati = daha güvenli → negatif katkı
        flag_score -= df[is_business_col].fillna(0) * 0.1

    score_components["flag_score"] = flag_score.clip(0, 1)

    # --- Bileşen 3: Velocity ---
    # Tüm entity'lerin velocity kolonlarını topla
    if velocity_1h_cols is None:
        velocity_1h_cols = [c for c in df.columns if c.endswith("_velocity_1h")]
    if velocity_1d_cols is None:
        velocity_1d_cols = [c for c in df.columns if c.endswith("_velocity_1d")]

    all_velocity_cols = velocity_1h_cols + velocity_1d_cols
    available_vel = [c for c in all_velocity_cols if c in df.columns]

    if available_vel:
        vel_matrix = df[available_vel].fillna(0).clip(0, velocity_cap) / velocity_cap
        # En yüksek velocity sinyali baskın olsun
        score_components["velocity_score"] = vel_matrix.max(axis=1).clip(0, 1)
    else:
        score_components["velocity_score"] = 0.0

    # --- Bileşik temporal skor ---
    # Saat profili: 0.35, flag'ler: 0.25, velocity: 0.40
    df["temporal_score"] = (
        score_components["hour_score"] * 0.35
        + score_components["flag_score"] * 0.25
        + score_components["velocity_score"] * 0.40
    ).clip(0, 1).round(4)

    # --- Driver'lar ---
    def get_temporal_drivers(row_idx):
        drivers = []

        if hour_col in df.columns:
            h = df.loc[row_idx, hour_col]
            if not pd.isna(h) and (h < 6 or h >= 22):
                drivers.append(f"tx_hour:{int(h)}")

        if is_night_col in df.columns and df.loc[row_idx, is_night_col] == 1:
            drivers.append("is_night:True")

        if is_weekend_col in df.columns and df.loc[row_idx, is_weekend_col] == 1:
            drivers.append("is_weekend:True")

        # En yüksek velocity
        for col in available_vel[:3]:
            val = df.loc[row_idx, col]
            if not pd.isna(val) and val > 3:
                drivers.append(f"{col}:{int(val)}")

        return ", ".join(drivers[:top_n_drivers]) if drivers else ""

    df["temporal_drivers"] = [get_temporal_drivers(i) for i in df.index]

    if verbose:
        print("=" * 60)
        print("TEMPORAL ANOMALI DETECTION")
        print("=" * 60)
        print(f"  Velocity kolonları         : {len(available_vel)} adet")
        print(f"  Score — mean               : {df['temporal_score'].mean():.4f}")
        print(f"  Score — max                : {df['temporal_score'].max():.4f}")
        high = (df["temporal_score"] > 0.5).sum()
        print(f"  Yüksek anomali (>0.5)      : {high:,} satır ({high/len(df)*100:.2f}%)")
        if target_col and target_col in df.columns:
            fraud_temporal = df[df[target_col] == 1]["temporal_score"].mean()
            legit_temporal = df[df[target_col] == 0]["temporal_score"].mean()
            print(f"  Fraud temporal mean        : {fraud_temporal:.4f}")
            print(f"  Legit temporal mean        : {legit_temporal:.4f}")
            print(f"  Ayrım gücü (fark)          : {fraud_temporal - legit_temporal:.4f}")
        print("=" * 60)

    return df



## 6. Orkestratör — `run_anomaly_detection()`

Tüm katmanları sırayla çalıştırır ve sonuçları birleştirir.

**Akış:**
```
df_features (Adım 3 çıktısı)
    ↓
[1/5] feature_selection_report()   → selected_features (48 kolon)
    ↓
[2/5] detect_column_anomalies()    → column_score_max, column_score_mean, column_drivers
    ↓
[3/5] detect_multivariate_anomalies() → multivariate_score  +  model kaydedilir
    ↓
[4/5] detect_entity_anomalies()    → entity_score, entity_drivers
    ↓
[5/5] detect_temporal_anomalies()  → temporal_score, temporal_drivers
    ↓
df_scored (590,540 × 509) + explanations dict
```

**Üretilen yeni kolonlar:** `column_score_max`, `column_score_mean`, `column_drivers`,  
`multivariate_score`, `entity_score`, `entity_drivers`, `temporal_score`, `temporal_drivers`  
→ **10 yeni kolon** (499 → 509)

In [7]:
# =============================================================================
# 6. ORKESTRATÖR
# =============================================================================

def run_anomaly_detection(
    df: pd.DataFrame,
    entity_cols: list = None,
    target_col: str = "isFraud",
    variance_threshold: float = 0.01,
    correlation_threshold: float = 0.95,
    contamination: float = 0.035,
    z_score_cap: float = 10.0,
    velocity_cap: float = 20.0,
    min_tx_count: int = 5,
    verbose: bool = True,
) -> tuple:
    """
    Multi-Layer Anomali Detection orkestratörü.

    Sıra:
        1. feature_selection_report
        2. detect_column_anomalies
        3. detect_multivariate_anomalies
        4. detect_entity_anomalies
        5. detect_temporal_anomalies

    Parameters
    ----------
    df                   : df_features (Adım 3 çıktısı)
    entity_cols          : Entity kolon listesi
    target_col           : Fraud etiketi (test'te None geç)
    variance_threshold   : Feature selection varyans eşiği
    correlation_threshold: Feature selection korelasyon eşiği
    contamination        : Isolation Forest kontaminasyon oranı
    z_score_cap          : Column anomali z-score üst sınırı
    velocity_cap         : Velocity normalize üst sınırı
    min_tx_count         : Entity fallback eşiği

    Returns
    -------
    df_scored    : Tüm anomali skorları eklenmiş dataframe
    explanations : {TransactionID → driver dict} açıklama sözlüğü
    fs_report    : feature_selection_report çıktısı (Adım 5'te kullanılacak)
    """

    if entity_cols is None:
        entity_cols = ["card1", "P_emaildomain", "DeviceType"]

    if verbose:
        print("\n" + "=" * 60)
        print("RUN ANOMALY DETECTION — BAŞLIYOR")
        print(f"  Satır sayısı  : {len(df):,}")
        print(f"  Kolon sayısı  : {len(df.columns):,}")
        print(f"  Entity kolonları: {entity_cols}")
        print("=" * 60 + "\n")

    # Step 1 — Feature Selection
    if verbose:
        print("[1/5] Feature Selection...")
    fs_report = feature_selection_report(
        df=df,
        target_col=target_col,
        variance_threshold=variance_threshold,
        correlation_threshold=correlation_threshold,
        verbose=verbose,
    )

    # Step 2 — Column Anomalies
    if verbose:
        print("\n[2/5] Column Anomalies...")
    df = detect_column_anomalies(
        df=df,
        target_col=target_col,
        z_score_cap=z_score_cap,
        verbose=verbose,
    )

    # Step 3 — Multivariate Anomalies
    if verbose:
        print("\n[3/5] Multivariate Anomalies (Isolation Forest)...")
    df = detect_multivariate_anomalies(
        df=df,
        selected_features=fs_report["selected_features"],
        contamination=contamination,
        verbose=verbose,
    )

    # Step 4 — Entity Anomalies
    if verbose:
        print("\n[4/5] Entity Anomalies...")
    df = detect_entity_anomalies(
        df=df,
        entity_cols=entity_cols,
        target_col=target_col,
        min_tx_count=min_tx_count,
        velocity_cap=velocity_cap,
        verbose=verbose,
    )

    # Step 5 — Temporal Anomalies
    if verbose:
        print("\n[5/5] Temporal Anomalies...")
    df = detect_temporal_anomalies(
        df=df,
        target_col=target_col,
        velocity_cap=velocity_cap,
        verbose=verbose,
    )

    # --- Açıklama sözlüğü ---
    id_col = "TransactionID" if "TransactionID" in df.columns else df.index.name or "index"

    explanations = {}
    for idx, row in df.iterrows():
        tx_id = row.get(id_col, idx) if id_col != "index" else idx
        explanations[tx_id] = {
            "column_score_max": row.get("column_score_max", 0),
            "column_score_mean": row.get("column_score_mean", 0),
            "column_drivers": row.get("column_drivers", ""),
            "multivariate_score": row.get("multivariate_score", 0),
            "entity_score": row.get("entity_score", 0),
            "entity_drivers": row.get("entity_drivers", ""),
            "temporal_score": row.get("temporal_score", 0),
            "temporal_drivers": row.get("temporal_drivers", ""),
        }

    # --- Final özet ---
    score_cols = [
        "column_score_max", "column_score_mean",
        "multivariate_score", "entity_score", "temporal_score"
    ]

    if verbose:
        print("\n" + "=" * 60)
        print("ANOMALI DETECTION TAMAMLANDI — ÖZET")
        print("=" * 60)
        for col in score_cols:
            if col in df.columns:
                print(f"  {col:<25}: mean={df[col].mean():.4f}  max={df[col].max():.4f}")

        if target_col and target_col in df.columns:
            print()
            print("  Fraud vs Legit karşılaştırması:")
            fraud = df[df[target_col] == 1]
            legit = df[df[target_col] == 0]
            for col in score_cols:
                if col in df.columns:
                    f_mean = fraud[col].mean()
                    l_mean = legit[col].mean()
                    lift = f_mean / max(float(l_mean), 1e-6) if l_mean > 0 else 0
                    print(f"    {col:<25}: fraud={f_mean:.4f}  legit={l_mean:.4f}  lift={lift:.2f}x")
        print("=" * 60)

    return df, explanations, fs_report



## 7. Çalıştır

**Girdi:** `step3_features.parquet` (Adım 3)  
**Çıktı:** `step4_scored.parquet` (590,540 × 509)

Beklenen çıktı özeti:
```
column_score_max  : mean=0.4924  max=1.0000  lift=1.47x
column_score_mean : mean=0.0318  max=0.2882  lift=1.64x
multivariate_score: mean=0.3034  max=1.0000  lift=1.26x
entity_score      : mean=0.0118  max=0.0772  lift=1.07x
temporal_score    : mean=0.5197  max=0.6667  lift=1.03x
```

> Isolation Forest modeli otomatik olarak `pipeline/models/` altına kaydedilir.

In [8]:
# =============================================================================
# KULLANIM ÖRNEĞİ
# =============================================================================

if __name__ == "__main__":
    import pandas as pd

    # Adım 3 çıktısını yükle
    df_features = pd.read_parquet("../Adım 3/step3_features.parquet")
    print(f"Yüklendi: {df_features.shape}")

    # Anomali detection çalıştır
    df_scored, explanations, fs_report = run_anomaly_detection(
        df=df_features,
        entity_cols=["card1", "P_emaildomain", "DeviceType"],
        target_col="isFraud",   # test setinde None geç
    )

    # Örnek açıklama
    sample_id = list(explanations.keys())[0]
    print(f"\nÖrnek açıklama (ID: {sample_id}):")
    for k, v in explanations[sample_id].items():
        print(f"  {k}: {v}")

    # Adım 5 için kaydet
    df_scored.to_parquet("step4_scored.parquet", index=False)
    print(f"\nKaydedildi: step4_scored.parquet — {df_scored.shape}")

Yüklendi: (590540, 499)

RUN ANOMALY DETECTION — BAŞLIYOR
  Satır sayısı  : 590,540
  Kolon sayısı  : 499
  Entity kolonları: ['card1', 'P_emaildomain', 'DeviceType']

[1/5] Feature Selection...
FEATURE SELECTION REPORT
  Başlangıç feature sayısı : 466
  Düşük varyans elenen      : 409
  Yüksek korelasyon elenen  : 9
  Seçilen feature sayısı    : 48

  [Düşük Varyans] İlk 5: ['TransactionAmt', 'card3', 'addr2', 'dist1', 'dist2']
  [Yüksek Korelasyon] İlk 5:
    - TransactionID: corr=0.998 with TransactionDT
    - TransactionDT: corr=1.000 with tx_days_since_start
    - devicetype_tx_count: corr=1.000 with devicetype_amt_mean
    - devicetype_amt_mean: corr=1.000 with devicetype_amt_std
    - devicetype_amt_std: corr=1.000 with devicetype_amt_median

[2/5] Column Anomalies...
COLUMN ANOMALI DETECTION
  Değerlendirilen kolon     : 466
  column_score_max  — mean  : 0.4924
  column_score_max  — max   : 1.0000
  column_score_mean — mean  : 0.0318
  Yüksek anomali (>0.5)     : 244,064 satır 

---

## Özet

| Katman | Skor | Lift | Güç | Not |
|--------|------|------|-----|-----|
| Column (max) | `column_score_max` | 1.47x | ⭐⭐⭐ | Tek kolon sapması |
| Column (mean) | `column_score_mean` | 1.64x | ⭐⭐⭐⭐ | Çok kolon sapması — en güçlü |
| Multivariate | `multivariate_score` | 1.26x | ⭐⭐⭐ | Joint distribution |
| Entity | `entity_score` | 1.07x | ⭐⭐ | Davranışsal baseline |
| Temporal | `temporal_score` | 1.03x | ⭐ | Zaman pattern |

**Kritik bulgu:** `column_score_mean` en yüksek fraud lift değerine sahip.  
Adım 5'te AUC bazlı ağırlıklandırma bu sıralamayı doğrulayacak.

**Sonraki adım:** Adım 5 — Score Aggregation  
Girdi: `step4_scored.parquet`